# Домашнее задание № 5
**Тема 6. Классы и объекты (ООП)**  
**Тема работы:** Банковский счёт и дополнительные механизмы безопасности  
**Курс:** Программирование на Python, 1 курс

**Цель:** реализовать обычный и кредитный банковские счета, сохранять историю операций и проверить дополнительные механизмы контроля операций.

В работе сначала реализованы обязательные требования, затем — дополнительные функции. Все операции выполняются в учебной модели, без подключения к настоящему банку.

## 1. Основной класс `Account`

Объект хранит владельца, баланс и список операций. Баланс находится в атрибуте `_balance`; для чтения используется метод `get_balance()`. Каждая попытка пополнения или снятия записывается в историю, включая неуспешные попытки. Для финансовых вычислений внутри класса используется `Decimal`, чтобы не накапливать ошибки двоичной арифметики с дробными суммами.

In [ ]:
from datetime import datetime
from decimal import Decimal, InvalidOperation


class Account:
    """Учебная модель обычного банковского счёта."""

    def __init__(self, account_holder, balance=0):
        if not isinstance(account_holder, str) or not account_holder.strip():
            raise ValueError("Имя владельца должно быть непустой строкой")

        initial_balance = self._to_money(balance)
        if initial_balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder = account_holder.strip()
        self._balance = initial_balance
        self.operations_history = []

    @staticmethod
    def _to_money(amount):
        """Проверяет числовое значение и округляет его до копеек."""
        if isinstance(amount, bool):
            raise ValueError("Сумма должна быть числом")
        try:
            value = Decimal(str(amount))
        except (InvalidOperation, ValueError, TypeError):
            raise ValueError("Сумма должна быть числом") from None
        if not value.is_finite():
            raise ValueError("Сумма должна быть конечным числом")
        if value != value.quantize(Decimal("0.01")):
            raise ValueError("Допускается не более двух знаков после запятой")
        return value

    def _record_operation(self, operation_type, amount, status, reason=None):
        """Добавляет структурированную запись об операции."""
        record = {
            "type": operation_type,
            "amount": float(amount),
            "datetime": datetime.now(),
            "balance_after": float(self._balance),
            "status": status,
        }
        if reason is not None:
            record["reason"] = reason
        self.operations_history.append(record)

    def deposit(self, amount):
        amount = self._to_money(amount)
        if amount <= 0:
            self._record_operation("deposit", amount, "fail", "Неверная сумма")
            return False
        self._balance += amount
        self._record_operation("deposit", amount, "success")
        return True

    def withdraw(self, amount):
        amount = self._to_money(amount)
        if amount <= 0:
            self._record_operation("withdraw", amount, "fail", "Неверная сумма")
            return False
        if amount > self._balance:
            self._record_operation("withdraw", amount, "fail", "Недостаточно средств")
            return False
        self._balance -= amount
        self._record_operation("withdraw", amount, "success")
        return True

    def get_balance(self):
        return float(self._balance)

    def get_history(self):
        """Возвращает копии записей, защищая историю от внешних изменений."""
        return [record.copy() for record in self.operations_history]

    def get_statistics(self):
        """Дополнительная аналитика по выполненным операциям."""
        history = self.get_history()
        return {
            "deposits_total": round(sum(
                op["amount"] for op in history
                if op["type"] == "deposit" and op["status"] == "success"
            ), 2),
            "withdrawals_total": round(sum(
                op["amount"] for op in history
                if op["type"] == "withdraw" and op["status"] == "success"
            ), 2),
            "successful_operations": sum(op["status"] == "success" for op in history),
            "failed_operations": sum(op["status"] == "fail" for op in history),
        }


### Проверка обычного счёта

Проверяем пополнение, успешное снятие и отказ при недостатке средств.

In [ ]:
account = Account("Алексей", 10000)
print("Начальный баланс:", account.get_balance())
print("Пополнение на 2500:", account.deposit(2500))
print("Снятие 3000:", account.withdraw(3000))
print("Попытка снять 20000:", account.withdraw(20000))
print("Итоговый баланс:", account.get_balance())
print("Статистика:", account.get_statistics())
for operation in account.get_history():
    print(operation)

## 2. Наследование: `CreditAccount`

Кредитный счёт наследует методы обычного счёта. Переопределяется снятие денег: разрешён отрицательный баланс, но не ниже `-credit_limit`. В историю добавляется поле `credit_used` — использовались ли кредитные средства в конкретной операции. Метод `get_available_credit()` показывает, сколько ещё можно потратить с учётом собственных средств и лимита.

In [ ]:
class CreditAccount(Account):
    """Банковский счёт с возможностью использовать кредитный лимит."""

    def __init__(self, account_holder, balance=0, credit_limit=0):
        super().__init__(account_holder, 0)
        self._credit_limit = self._to_money(credit_limit)
        if self._credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным")

        initial_balance = self._to_money(balance)
        if initial_balance < -self._credit_limit:
            raise ValueError("Начальный долг превышает кредитный лимит")
        self._balance = initial_balance

    def _record_operation(self, operation_type, amount, status, reason=None):
        super()._record_operation(operation_type, amount, status, reason)
        # Для снятия отмечаем использование кредита только при успехе.
        used_credit = (
            operation_type == "withdraw"
            and status == "success"
            and self._balance < 0
        )
        self.operations_history[-1]["credit_used"] = used_credit

    def withdraw(self, amount):
        amount = self._to_money(amount)
        if amount <= 0:
            self._record_operation("withdraw", amount, "fail", "Неверная сумма")
            return False
        if self._balance - amount < -self._credit_limit:
            self._record_operation("withdraw", amount, "fail", "Превышен кредитный лимит")
            return False
        self._balance -= amount
        self._record_operation("withdraw", amount, "success")
        return True

    def get_available_credit(self):
        return float(self._balance + self._credit_limit)


In [ ]:
credit_account = CreditAccount("Мария", balance=1000, credit_limit=5000)
print("Доступные средства:", credit_account.get_available_credit())
print("Снятие 4000:", credit_account.withdraw(4000))
print("Баланс после снятия:", credit_account.get_balance())
print("Осталось доступно:", credit_account.get_available_credit())
print("Снятие 3000:", credit_account.withdraw(3000))
for operation in credit_account.get_history():
    print(operation)

## 3. Дополнительная часть: контроль подозрительных операций

В качестве расширения создан класс `SecureAccount`. Он наследует `Account` и добавляет:

- ограничение суммы одного снятия;
- блокировку снятий после трёх подряд неудачных попыток;
- отдельный журнал событий безопасности;
- демонстрационную разблокировку методом `unblock_account()`.

**Ограничение модели:** это учебная логика, а не реальная банковская защита. В настоящей системе разблокировка требует проверки полномочий пользователя, а журналы должны храниться в защищённом внешнем хранилище.

In [ ]:
class SecureAccount(Account):
    """Обычный счёт с учебными механизмами контроля снятий."""

    def __init__(self, account_holder, balance=0, withdrawal_limit=50000):
        super().__init__(account_holder, balance)
        self._withdrawal_limit = self._to_money(withdrawal_limit)
        if self._withdrawal_limit <= 0:
            raise ValueError("Лимит снятия должен быть положительным")
        self._failed_attempts = 0
        self._blocked = False
        self._security_log = []

    def _log_security(self, event, details):
        self._security_log.append({
            "datetime": datetime.now(),
            "event": event,
            "details": details,
        })

    def withdraw(self, amount):
        amount = self._to_money(amount)

        if self._blocked:
            self._record_operation("withdraw", amount, "fail", "Счёт заблокирован")
            self._log_security("blocked_attempt", "Попытка снятия с заблокированного счёта")
            return False

        if amount > self._withdrawal_limit:
            self._record_operation("withdraw", amount, "fail", "Превышен разовый лимит")
            self._log_security("large_withdrawal", "Превышен лимит одной операции")
            self._register_failure()
            return False

        success = super().withdraw(amount)
        if success:
            self._failed_attempts = 0
        else:
            self._register_failure()
        return success

    def _register_failure(self):
        self._failed_attempts += 1
        if self._failed_attempts >= 3:
            self._blocked = True
            self._log_security("account_blocked", "Три неудачных снятия подряд")

    def unblock_account(self):
        """Учебная разблокировка без проверки личности."""
        self._blocked = False
        self._failed_attempts = 0
        self._log_security("account_unblocked", "Демонстрационная разблокировка")

    def is_blocked(self):
        return self._blocked

    def get_security_log(self):
        return [event.copy() for event in self._security_log]


In [ ]:
secure = SecureAccount("Ирина", balance=10000, withdrawal_limit=50000)
for attempt in range(1, 4):
    print(f"Попытка {attempt}:", secure.withdraw(20000))
print("Заблокирован:", secure.is_blocked())
print("Пополнение доступно:", secure.deposit(1000))
print("Снятие при блокировке:", secure.withdraw(100))
print("События безопасности:")
for event in secure.get_security_log():
    print(event)
secure.unblock_account()
print("После разблокировки:", secure.is_blocked())

## 4. Автоматическая проверка

Проверяем корректность баланса, истории, времени операций, наследования и блокировки. Если тесты завершатся без ошибок, появится сообщение об успешной проверке.

In [ ]:
# Обычный счёт
basic = Account("Тест", 100)
assert basic.deposit(25.50) is True
assert basic.withdraw(30) is True
assert basic.get_balance() == 95.5
assert basic.withdraw(1000) is False
assert basic.deposit(-5) is False
assert basic.get_history()[-1]["status"] == "fail"
assert isinstance(basic.get_history()[0]["datetime"], datetime)
assert basic.get_statistics()["failed_operations"] == 2

# Внешнее изменение истории не меняет исходные записи.
history_copy = basic.get_history()
history_copy[0]["status"] = "changed"
assert basic.get_history()[0]["status"] == "success"

# Проверка некорректного начального баланса.
try:
    Account("Тест", -1)
    assert False, "Ожидалась ошибка ValueError"
except ValueError:
    pass

# Кредитный счёт: можно уйти в минус, но нельзя превысить лимит.
credit = CreditAccount("Тест", balance=100, credit_limit=200)
assert credit.withdraw(250) is True
assert credit.get_balance() == -150
assert credit.get_available_credit() == 50
assert credit.get_history()[-1]["credit_used"] is True
assert credit.withdraw(51) is False
assert credit.get_balance() == -150
assert credit.deposit(100) is True
assert credit.get_balance() == -50

# Дополнительная защита: три отказа приводят к блокировке.
protected = SecureAccount("Тест", balance=100)
for _ in range(3):
    assert protected.withdraw(200) is False
assert protected.is_blocked() is True
assert protected.withdraw(1) is False
assert protected.deposit(10) is True
assert any(e["event"] == "account_blocked" for e in protected.get_security_log())
protected.unblock_account()
assert protected.is_blocked() is False
assert protected.withdraw(10) is True

print("Все проверки пройдены успешно!")

## 5. Вывод

В ходе работы реализованы классы `Account` и `CreditAccount`. Программа позволяет пополнять счёт, снимать средства, получать баланс и просматривать историю успешных и неуспешных операций с датой и временем. Для кредитного счёта учитывается доступный лимит и использование кредитных средств.

В дополнительной части реализованы простые механизмы контроля снятий и журнал безопасности. Также добавлена статистика операций и автоматические тесты. Работа помогла закрепить конструкторы классов, методы, наследование и обработку ошибок.